In [34]:
import pandas as pd

df = pd.read_csv("../data/raw/genetic_disorder_raw.csv")
identity_cols = ["Patient Id", "Patient First Name", "Family Name", "Father's name"]
df_clean = df.drop(columns=identity_cols)

numeric_cols = df_clean.select_dtypes(include=["float64", "int64"]).columns
categorical_cols = df_clean.select_dtypes(include="object").columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in categorical_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

In [19]:
# Load the cleaned dataset
import pandas as pd 
df = pd.read_csv("../data/processed/genetic_disorder_encoded.csv")
df.shape 

(18000, 29)

In [20]:
# Separate features (X) from target (Y)
# "Feature" = the input clues. "Target" = what you're trying to predict
# Drop Disorder Subclass too — we're predicting Genetic Disorder only for now
X = df.drop(columns=["Genetic Disorder", "Disorder Subclass"])
y = df["Genetic Disorder"]

X.shape, y.shape

((18000, 27), (18000,))

In [21]:
# Split data into train and test sets
# Train set = what the model learns from. 
# Test set = unseen data to check if it actually learned, not memorized.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((14400, 27), (3600, 27))

In [22]:
# Train a basic Random Forest (first pass, no tuning yet)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
# Check accuracy on the test set 
from sklearn.metrics import accuracy_score

y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

0.6741666666666667

In [24]:
# Check confusion matrix - to see WHERE it's struggling
from sklearn.metrics import confusion_matrix, classification_report 

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 763  107  330]
 [  25 1142   33]
 [ 489  189  522]]
              precision    recall  f1-score   support

           0       0.60      0.64      0.62      1200
           1       0.79      0.95      0.87      1200
           2       0.59      0.43      0.50      1200

    accuracy                           0.67      3600
   macro avg       0.66      0.67      0.66      3600
weighted avg       0.66      0.67      0.66      3600



In [25]:
# Check feature importance - to see WHAT the model is actually using 
import pandas as pd 

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(10)

Blood cell count (mcL)                              0.097020
White Blood cell count (thousand per microliter)    0.088196
Father's age                                        0.075937
Mother's age                                        0.073263
Patient Age                                         0.069859
Symptom 5                                           0.059598
Symptom 4                                           0.043736
Symptom 3                                           0.039790
H/O radiation exposure (x-ray)                      0.039083
H/O substance abuse                                 0.038996
dtype: float64

In [26]:
# hyperparameter tuning - to see if we can improve accuracy
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
0.6547916666666667


In [27]:
# Train final model with best params, check test accuracy 
final_rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=None,
    min_samples_split=2,
    random_state=42
)
final_rf.fit(X_train, y_train)

y_pred_final = final_rf.predict(X_test)
print("Final accuracy:", accuracy_score(y_test, y_pred_final))

Final accuracy: 0.6752777777777778


In [28]:
# Try feature engineering - create new combined features instead of just raw columns
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# Total symptom count instead of 5 separate columns 
symptom_cols = ["Symptom 1", "Symptom 2", "Symptom 3", "Symptom 4", "Symptom 5"]
X_train_fe["Symptom_Count"] = X_train_fe[symptom_cols].sum(axis=1)
X_test_fe["Symptom_Count"] = X_test_fe[symptom_cols].sum(axis=1)

# Parental age gap 
X_train_fe["Parent_Age_Gap"] = X_train_fe["Father's age"] - X_train_fe["Mother's age"]
X_test_fe["Parent_Age_Gap"] = X_test_fe["Father's age"] - X_test_fe["Mother's age"]

In [38]:
final_rf_fe = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    random_state=42
)
final_rf_fe.fit(X_train_fe, y_train)

y_pred_fe = final_rf_fe.predict(X_test_fe)
print("Final model Accuracy:", accuracy_score(y_test, y_pred_fe))

Final model Accuracy: 0.685


In [30]:
# Gradient Boosting Classifier - to see if a different model performs better
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_fe, y_train)
y_pred_gb = gb.predict(X_test_fe)
print("Gradient Boosting accuracy:", accuracy_score(y_test, y_pred_gb))

Gradient Boosting accuracy: 0.6133333333333333


In [31]:
# drop low_importance/noisy columns 
importances_fe = pd.Series(final_rf_fe.feature_importances_, index=X_train_fe.columns)
low_importance = importances_fe[importances_fe < 0.01].index
print("Dropping:", list(low_importance))

X_train_reduced = X_train_fe.drop(columns=low_importance)
X_test_reduced = X_test_fe.drop(columns=low_importance)

rf_reduced = RandomForestClassifier(n_estimators=200, random_state=42)
rf_reduced.fit(X_train_reduced, y_train)
y_pred_reduced = rf_reduced.predict(X_test_reduced)
print("Accuracy after removing low-importance columns:", accuracy_score(y_test, y_pred_reduced))

Dropping: []
Accuracy after removing low-importance columns: 0.685


In [35]:
# fix the encoding with One-Hot Encoding for categorical variables
categorical_cols = df_clean.select_dtypes(include="object").columns
categorical_cols_no_target = [c for c in categorical_cols if c not in ["Genetic Disorder", "Disorder Subclass"]]

df_onehot = pd.get_dummies(df_clean, columns=categorical_cols_no_target, drop_first=True)
df_onehot.shape

(18000, 37)

In [36]:
# redo the split and train 
X2 = df_onehot.drop(columns=["Genetic Disorder", "Disorder Subclass"])
y2 = df_onehot["Genetic Disorder"]

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

rf_oh = RandomForestClassifier(n_estimators=200, random_state=42)
rf_oh.fit(X2_train, y2_train)
print("One-hot accuracy:", accuracy_score(y2_test, rf_oh.predict(X2_test)))

One-hot accuracy: 0.6811111111111111


In [37]:
# trade-off include Disorder Subclass as a training feature for predicting Genetic Disorder 
X3 = df_onehot.drop(columns=["Genetic Disorder"])
if "Disorder Subclass" in X3.columns and X3["Disorder Subclass"].dtype == "object":
    X3 = pd.get_dummies(X3, columns=["Disorder Subclass"], drop_first=True)

y3 = df_onehot["Genetic Disorder"]

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)

rf3 = RandomForestClassifier(n_estimators=200, random_state=42)
rf3.fit(X3_train, y3_train)
print("Accuracy with Disorder Subclass included:", accuracy_score(y3_test, rf3.predict(X3_test)))

Accuracy with Disorder Subclass included: 0.95


In [39]:
# Save real final model 
import joblib 
joblib.dump(final_rf_fe, "../model/registry/model_v1.pkl")

['../model/registry/model_v1.pkl']